## Import libraries

In [ ]:
# general
import sys, os
from os.path import join
from os import path
from importlib import reload
from getpass import getuser
from tqdm.auto import tqdm
import gc
import json

# Data
import xarray as xr
import numpy as np
import imageio
from collections import Counter

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

# skimage
import skimage.morphology

# scipy
from scipy.ndimage import gaussian_filter, zoom

# pyFAI
from pyFAI.integrator.azimuthal import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(join(os.getcwd(), "library"))
import helper_functions as helper
import mask_lib
import interactive
from interactive import cimshow
import phase_retrieval_core as PhR

plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Self-written library, uses the GPU automatically if available
import CCI_core as cci

GPU = cci.GPU
print("GPU available" if GPU else "GPU unavailable")
parula_map = cci.parula_map()

In [ ]:
# interactive plotting
import ipywidgets

%matplotlib widget

# Auto formatting of cells
#%load_ext jupyter_black

# Setup

## Definitions

Beamtime-specific settings: facility, data and output folders, file naming (`scan_folder_name`, `date_time`, `ccd_prefix`) and `det_dist_offset`. **Adapt at the start of each beamtime.**

In [ ]:
facility = "MAXIV" # Options: "MAXIV", "PETRA", "MAXI", "SwissFEL"
scan_folder_name = "contrast" # subfolder inside raw folder
date_time = 20260930 # filename (date) of the h5 file with single images (not taken with scan macros)
ccd_prefix = "alignment_" # filename prefix of ccd files with single images
det_dist_offset = 65 # mm, subtracted from the det_dist motor position to get the sample-detector distance
USER = getuser()

# Facility specific loading functions
if facility == "SwissFEL":
    from sfdata import SFDataFiles, SFScanInfo, SFProcFile
    from loaders import Swiss_FEL_Loading as loading
else:
    # MAXI, PETRA and MAXIV share the same hdf5 loader
    from loaders import hdf5_loading as loading

BASEFOLDER = "/data/visitors/softimax/20260958/2026093008"
DATAFOLDER = join(BASEFOLDER,"raw")

# General saving folder and log folder
folder_general = helper.create_folder(join(BASEFOLDER, "process","Imaging"))
folder_log = helper.create_folder(join(folder_general,"Log"))

print("Raw Datafolder is: %s"%DATAFOLDER)
print("Output Folder: %s" % folder_general)
print("Log file Folder: %s" % folder_log)


# Load dictionary for keys etc
if facility == "SwissFEL":
    mnemonics = loading.load_mnemonics()
else:
    mnemonics = loading.load_mnemonics(facility)

## Experiment specific functions

Functions for loading data, log files, the FTH worker for batch processing, and plotting. Run once; only adapt them if the data format changes.

### Log files

Saving and loading of reconstructions (hdf5) and of mask, support and ROI coordinates (json) in `folder_log`. Used by *3. Option B* to resume an analysis.

In [ ]:
# Reconstruction log files (hdf5)
def save_cdi_h5(fname):
    """Basic Saving of global variables"""
    
    # Save h5
    data = {}
    data["im_id"] = im_id
    data["topo_id"] = topo_id
    data["roi_im_reg"] = roi_im_reg
    data["pos"] = pos.astype("float32")
    data["neg"] = neg.astype("float32")
    data["factor"] = factor
    data["offset"] = offset
    data["center"] = center
    data["roi"] = roi_cdi
    data["prop_dist"] = prop_dist
    data["prop_dist_cdi"] = prop_dist_cdi
    data["phase"] = phase_cdi
    data["roi_fth"] = roi
    data["phase_fth"] = phase
    data["sample"] = sample
    data["supportmask"] = supportmask.astype(bool)
    data["mask_pixel"] = mask_pixel.astype(bool)
    data["p_pc"] = p_pc.astype("complex64")
    data["n_pc"] = n_pc.astype("complex64")
    data["experimental_setup"] = experimental_setup
    data["shift"] = shift
    data["phase_retrieval_recipe"] = phase_retrieval_recipe

    filename = join(
        folder_log,
        fname,
    )
    print("Now Saving: %s" % fname)
    loading.create_hdf5(data, filename)
    return

def read_cdi_h5(im_id, topo_id, USER):
    """Load h5 reconstruction logfiles"""
    filename = join(
        folder_log,
        "Data_ImId_%s_RefId_%s_cdi_%s.hdf5" % (im_id, topo_id, USER),
    )
    print("Now Loading: %s" % filename)
    data = loading.read_hdf5(filename)
    return data


In [ ]:
# Load all parameters of a reconstruction from its log file
def load_reconstruction_params(im_id, topo_id, USER):
    """Load all important reconstruction parameters required for the notebook from logfiles"""
    data = read_cdi_h5(im_id, topo_id, USER)

    # Important parameters to load
    experimental_setup = data["experimental_setup"]
    sample = data["sample"].decode("utf-8")
    center = data["center"]
    shift = data["shift"]
    roi_im_reg = data["roi_im_reg"]
    mask_pixel = data["mask_pixel"]
    pos = data["pos"]

    # Reco specific
    roi_cdi = data["roi"]
    roi = data["roi_fth"]
    prop_dist = data["prop_dist"]
    prop_dist_cdi = data["prop_dist_cdi"]
    phase = data["phase_fth"]
    phase_cdi = data["phase"]
    supportmask = data["supportmask"]

    # h5py returns arrays and bytes: convert recipe back to lists, str and int
    phase_retrieval_recipe = {}
    for key, value in data["phase_retrieval_recipe"].items():
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, list):
            value = [v.decode("utf-8") if isinstance(v, bytes) else v for v in value]
        elif isinstance(value, bytes):
            value = value.decode("utf-8")
        phase_retrieval_recipe[key] = value
    # Other processing of data
    roi_im_reg_s = np.s_[roi_im_reg[0] : roi_im_reg[1], roi_im_reg[2] : roi_im_reg[3]]
    roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
    roi_cdi_s = np.s_[roi_cdi[0] : roi_cdi[1], roi_cdi[2] : roi_cdi[3]]
    
    footprint = skimage.morphology.disk(3 * 2)
    mask_pixel_smooth = skimage.morphology.dilation(mask_pixel, footprint)
    mask_pixel_smooth = gaussian_filter(mask_pixel_smooth, 2)
    mask_bs_cdi = np.ones(pos.shape)

    return experimental_setup, sample, pos, center, roi_im_reg, roi_im_reg_s, shift, mask_pixel, mask_pixel_smooth, roi, roi_s, roi_cdi, roi_cdi_s, prop_dist, prop_dist_cdi, phase, phase_cdi, supportmask, phase_retrieval_recipe, mask_bs_cdi
    

### Fitting

In [ ]:
from scipy.optimize import curve_fit
def cos2(x, a, b, c):
    return a * np.cos( x + b)**2 + c

# Load data

In [ ]:
im_id_set = [295,299,305,307]
topo_id_set = [297,301,303,309]

angles = [ 0, 90, 45, -45, 30, -30, 60, -60]

print("Angle positive Set:  %s" % im_id_set)
print("Angle negative Set: %s" % topo_id_set)
print("Dark Id positive Set: %s" % dark_id_im_set)
print("Dark Id negative Set: %s" % dark_id_topo_set)
print("Measurement angles: %s" %angles)

In [ ]:
# Load all data into lists
p_pc_list, n_pc_list, data_list = [], [], []
for im_id, topo_id in zip(im_id_set, topo_id_set):
    data = read_cdi_h5(im_id,topo_id,USER)
    p_pc_list.append(data["p_pc"])
    n_pc_list.append(data["n_pc"])
    data_list.append(data["p_pc"])
    data_list.append(data["n_pc"])
    
p_pc_list = np.stack(p_pc_list)
n_pc_list = np.stack(n_pc_list)
data_list = np.stack(data_list)
angles = np.array(angles)

supportmask = data["supportmask"]
roi = data["roi"]
roi_s = np.s_[roi[0]:roi[1],roi[2]:roi[3]]

# Crop to roi

p_pc_list = p_pc_list[:,*roi_s] 
n_pc_list = n_pc_list[:,*roi_s]
data_list = np.real(data_list[:,*roi_s])

supportmask = supportmask[roi_s]
data_list = data_list*supportmask
data = np.real((p_pc_list+n_pc_list))*supportmask

In [ ]:
fig, ax = cimshow(data*supportmask, cmap="gray")
fig.set_size_inches([4,4])

In [ ]:
fig, ax = cimshow(np.sum(data,axis=0)*supportmask, cmap="gray")
fig.set_size_inches([4,4])

# Fitting

## Test fit

In [ ]:
# Select pixel values
y = np.real(data_list[:,77,65])
data_mask = (angles < 500) + (angles > 65)
rad = np.deg2rad(angles)

# Fitting
a0 = np.max(y)-np.min(y)
b0 = 0
c0 = a0/2
BOUNDS = np.array([(0, 1e8), (-np.pi, np.pi), (-1000000, 1000000)]).T
popt, cov = curve_fit(cos2, rad[data_mask], y[data_mask], p0=(a0, b0, c0), bounds=BOUNDS)
rad_2 = np.linspace(-1, 1.5, 100)

# Plot
fig, ax = plt.subplots()
ax.plot(np.rad2deg(rad), y, ".", label="Data")
ax.plot(np.rad2deg(rad_2), cos2(rad_2, *popt), label = f"a*cos(x+b)+c\na={popt[0]:.2f}\nb={popt[1]:.2f}\nc={popt[2]:.2f}")
ax.legend()

In [ ]:
def init(d, r, m):
    global DATA, RAD, MASK
    DATA, RAD, MASK = d, r, m

def worker(ix):
    P0 = (np.max(y,axis=0)-np.min(y,axis=0), 0, a0/2)
    out = np.full((DATA.shape[2], 3), np.nan)
    x = RAD[MASK]
    for iy in range(DATA.shape[2]):            # one row per task
        try:
            out[iy], _ = curve_fit(cos2, x, DATA[MASK, ix, iy],
                                   p0=P0[ix, iy], bounds=BOUNDS)
        except Exception:
            pass
    return ix, outa
    
from multiprocessing import Pool, cpu_count

BOUNDS = ([0, -np.pi, -1e6], [1e8, np.pi, 1e6])

def init(d, r, m):
    global DATA, RAD, MASK
    DATA, RAD, MASK = d, r, m

def worker(ix):
    ny = DATA.shape[2]
    out = np.full((ny, 3), np.nan)
    errors = []
    x = RAD[MASK]
    for iy in range(ny):
        y = DATA[MASK, ix, iy]
        valid = np.isfinite(y) & (y != 0)       # drop masked (0) and NaN points
        if valid.sum() <= 3:                    # masked pixel or too few points
            continue
        xv, yv = x[valid], y[valid]

        # per-pixel start values
        a_guess = yv.max() - yv.min()
        b_guess = np.clip(xv[np.argmax(yv)], -np.pi + 1e-6, np.pi - 1e-6)
        c_guess = yv.min()
        p0 = (max(a_guess, 1e-6), b_guess, c_guess)

        try:
            out[iy], _ = curve_fit(cos2, xv, yv, p0=p0, bounds=BOUNDS)
        except Exception as e:
            errors.append((ix, iy, repr(e)))
    return ix, out, errors

nx, ny = data_list.shape[1:]
fit = np.full((nx, ny, 3), np.nan)
all_errors = []
with Pool(cpu_count(), initializer=init, initargs=(data_list, rad, data_mask)) as pool:
    for ix, row, errs in tqdm(pool.imap(worker, range(nx)), total=nx):
        fit[ix] = row
        all_errors += errs

print(f"{len(all_errors)} failed fits")
print(all_errors[:5])

In [ ]:
cimshow(np.array(fit)[:,:,1], cmap="twilight_shifted")
cimshow(np.array(fit)[:,:,0], cmap="twilight_shifted")

In [ ]:
# --- Daten wie bei dir ---
Dat = np.array(fit)
amp = Dat[:, :, 0]*supportmask
phase = Dat[:, :, 1]*supportmask

#phase_norm = (phase % (2*np.pi)) / (2*np.pi)*2
phase_norm = (phase - np.nanmin(phase)) / (np.nanmax(phase) - np.nanmin(phase))
amp_norm = amp / np.nanmax(np.abs(amp))

hsv = np.zeros((*amp.shape, 3))
hsv[..., 0] = phase_norm
hsv[..., 1] = 1.0
hsv[..., 2] = amp_norm

rgb = hsv_to_rgb(hsv)

# --- Plot Hauptbild ---
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(rgb)
ax.set_title("Phase (Hue) + Amplitude (Value)")
ax.axis('off')

# --- Kreis-Legende erzeugen ---
N = 300
x = np.linspace(-1, 1, N)
y = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, y)

R = np.sqrt(X**2 + Y**2)
Phi = np.arctan2(Y, X)

mask = R <= 1

phase_leg = (Phi % (2*np.pi)) / (2*np.pi)
amp_leg = np.clip(R, 0, 1)

hsv_leg = np.zeros((N, N, 3))
hsv_leg[..., 0] = phase_leg
hsv_leg[..., 1] = 1.0
hsv_leg[..., 2] = amp_leg

rgb_leg = hsv_to_rgb(hsv_leg)

# außerhalb des Kreises weiß setzen
rgb_leg[~mask] = 1.0

# --- Legende einfügen ---
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

axins = inset_axes(ax, width="15%", height="15%", loc='lower right')
axins.imshow(rgb_leg, extent=(-1,1,-1,1))
axins.set_xticks([])
axins.set_yticks([])
axins.axis('off')
#axins.set_title("Orientation", fontsize=8)

# Save images
fname = join(
    folder_general,
    "VectorMap_ImId_%s_RefId_%s_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

# Save log
data_save = {"Amplitude":amp,
            "Phase":phase,
            "fit_data":Dat,
            "angles":angles,
            "topo_id":topo_id_set,
            "im_id":im_id_set}

filename = join(
        folder_log,
        "XMLD_map_ImId_%s_RefId_%s_%s"%(im_id_set,topo_id_set,USER),
    )
print("Now Saving: %s" % filename)
loading.create_hdf5(data_save, filename)